# Logistics KPI Analysis & Root Cause Investigation

**Portfolio Project – Supply Chain & Operations Analytics**  
Focus: Performance tracking, delay root-cause analysis, and actionable insights for Fulfillment / Rapid Asset Deployment environments.

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Style settings for clean, professional visuals
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
# Load the dataset
# Place the CSV in the data/ folder or update the path
DATA_PATH = '../data/smart_logistics_dataset.csv'

df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Quick Data Quality Check

In [ ]:
print('--- Info ---')
df.info()
print('\n--- Missing Values ---')
print(df.isnull().sum())
print('\n--- Basic Statistics ---')
df.describe().T

In [ ]:
# Convert timestamp and create useful time features
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Date'] = df['Timestamp'].dt.date
df['Month'] = df['Timestamp'].dt.month
df['Hour'] = df['Timestamp'].dt.hour

# Create a clear binary flag already exists, but we keep it explicit
df['Is_Delayed'] = df['Logistics_Delay'].astype(bool)

print('Date range:', df['Timestamp'].min(), '→', df['Timestamp'].max())

## 3. Core KPI Calculation

In [ ]:
def calculate_core_kpis(dataframe):
    """Calculate the main operational KPIs used in daily stand-ups."""
    kpis = {
        'Total Records': len(dataframe),
        'Delay Rate (%)': round(dataframe['Logistics_Delay'].mean() * 100, 1),
        'Avg Asset Utilization (%)': round(dataframe['Asset_Utilization'].mean(), 1),
        'Avg Waiting Time (min)': round(dataframe['Waiting_Time'].mean(), 1),
        'Median Waiting Time (min)': round(dataframe['Waiting_Time'].median(), 1),
        'Avg Inventory Level': round(dataframe['Inventory_Level'].mean(), 0),
        'Avg Demand Forecast': round(dataframe['Demand_Forecast'].mean(), 0),
    }
    return pd.Series(kpis)

kpi_summary = calculate_core_kpis(df)
print('=== CORE OPERATIONAL KPIs ===')
print(kpi_summary)

## 4. Shipment Status & Delay Overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Shipment Status distribution
status_counts = df['Shipment_Status'].value_counts()
axes[0].pie(status_counts, labels=status_counts.index, autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette('pastel'))
axes[0].set_title('Shipment Status Distribution')

# Delay Rate by Traffic Status
delay_by_traffic = df.groupby('Traffic_Status')['Logistics_Delay'].mean().sort_values(ascending=False) * 100
sns.barplot(x=delay_by_traffic.index, y=delay_by_traffic.values, ax=axes[1], palette='Reds_r')
axes[1].set_title('Delay Rate (%) by Traffic Status')
axes[1].set_ylabel('Delay Rate (%)')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

## 5. Root Cause Analysis – Delay Reasons

In [ ]:
# Focus only on delayed records for root cause
delayed = df[df['Logistics_Delay'] == 1].copy()

print(f'Total delayed records: {len(delayed)}')
print('\nDelay Reason breakdown:')
reason_counts = delayed['Logistics_Delay_Reason'].value_counts(dropna=False)
print(reason_counts)

plt.figure(figsize=(9, 5))
sns.countplot(data=delayed, y='Logistics_Delay_Reason', order=reason_counts.index, palette='flare')
plt.title('Root Cause of Logistics Delays')
plt.xlabel('Number of Delayed Shipments')
plt.ylabel('Delay Reason')
plt.tight_layout()
plt.show()

## 6. Asset Utilization & Waiting Time Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Utilization distribution
sns.histplot(df['Asset_Utilization'], bins=20, kde=True, ax=axes[0], color='steelblue')
axes[0].axvline(df['Asset_Utilization'].mean(), color='red', linestyle='--', label=f'Mean: {df["Asset_Utilization"].mean():.1f}%')
axes[0].set_title('Asset Utilization Distribution')
axes[0].set_xlabel('Utilization (%)')
axes[0].legend()

# Waiting Time vs Delay
sns.boxplot(data=df, x='Is_Delayed', y='Waiting_Time', ax=axes[1], palette='Set2')
axes[1].set_title('Waiting Time by Delay Status')
axes[1].set_xlabel('Delayed?')
axes[1].set_ylabel('Waiting Time (minutes)')
axes[1].set_xticklabels(['No', 'Yes'])

plt.tight_layout()
plt.show()

In [ ]:
# Utilization by Asset
util_by_asset = df.groupby('Asset_ID')['Asset_Utilization'].mean().sort_values()

plt.figure(figsize=(10, 5))
sns.barplot(x=util_by_asset.index, y=util_by_asset.values, palette='viridis')
plt.axhline(util_by_asset.mean(), color='red', linestyle='--', label=f'Overall Mean: {util_by_asset.mean():.1f}%')
plt.title('Average Asset Utilization by Truck')
plt.ylabel('Avg Utilization (%)')
plt.xlabel('Asset ID')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 7. Inventory vs Demand Forecast Gap

In [ ]:
df['Inventory_Gap'] = df['Inventory_Level'] - df['Demand_Forecast']

plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x='Demand_Forecast', y='Inventory_Level', hue='Is_Delayed', alpha=0.6)
plt.plot([100, 300], [100, 300], 'r--', label='Perfect Match Line')
plt.title('Inventory Level vs Demand Forecast')
plt.xlabel('Demand Forecast')
plt.ylabel('Inventory Level')
plt.legend(title='Delayed')
plt.tight_layout()
plt.show()

print('Average Inventory Gap (Inventory - Forecast):', round(df['Inventory_Gap'].mean(), 1))

## 8. Key Insights & Recommendations

### Main Findings

1. **Delay Rate is high (~56%)** – This is the most critical KPI that needs daily monitoring and escalation thresholds.
2. **Traffic conditions** have a strong relationship with delays. Heavy and Detour statuses show elevated delay rates.
3. **Waiting Time** is noticeably higher for delayed shipments → process bottlenecks exist beyond pure transportation time.
4. **Asset Utilization** averages around 80%, with variation across trucks → opportunity for better load balancing.
5. Inventory and Demand Forecast are not perfectly aligned, which can create downstream pressure on logistics.

### Recommended Actions (suitable for stand-ups & leadership updates)

- Define a clear **Delay Rate threshold** (e.g. 45%) and escalate when breached.
- Investigate the top delay reasons (Traffic / Weather) weekly and track improvement actions.
- Review low-utilization assets for reallocation or process redesign.
- Create a simple daily KPI snapshot (Delay Rate + Top 3 Reasons + Avg Waiting Time) for morning stand-ups.
- Improve coordination between inventory planning and transportation to reduce last-minute pressure.

---

*This analysis is designed to demonstrate structured KPI tracking, root-cause thinking, and clear communication of operational insights – core skills for Process Assistant and Logistics Operations roles.*